# C1.3 · Attacking evaluation itself

**Function C — Red Teaming and Security Research with AI → Red Teaming with AI**  ·  *Security of AI*

Builds on **[C1.2 · Red-teaming an agent: designing the campaign](https://spbreed.github.io/cyber-commons/lessons/C1.2.html)**.

| | |
|---|---|
| Tools used | Cyber Commons eval harness, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Game the B2.1 harness deliberately, then close the hole you used.

**Why a security engineer needs it.** If the eval can be fooled, the assurance is theatre. The control it builds is: eval gaming, sandbagging, contamination and judge manipulation as test cases.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Your evaluation is a control, and controls get attacked. A benchmark with a leaked key, a skewed class balance or a scorer that can be satisfied without solving anything is a control that reports itself green forever.

> **At CyberTravels.** The benchmark that says CyberTravels' review harness scores 0.9 is itself a control, and it gets attacked. A leaked key or a loose matcher makes it report green forever.

## 2 · The framework

```
   the benchmark is a control, so attack it

   leaked key      -> the score is a training metric
   skewed classes  -> a constant answer scores 0.875
   loose matching  -> wrong answers match on basename
   gameable oracle -> satisfied without solving anything

   an unattacked evaluation reports itself green forever
```

If you can make a harness score well without being good, so can the vendor whose
benchmark you are reading — and so can your own team, without meaning to.

Three exploits work on almost every published security-harness result:

1. **Report conformance as quality.** Schema validity is ~100% by construction
   with structured output. It measures nothing about correctness (B2.1).
2. **Exploit class imbalance.** If 80% of a corpus is one CWE, always guessing
   that CWE scores 0.8 with no capability at all.
3. **Exploit basename collisions.** If the matcher compares bare filenames and
   the corpus reuses `1.py` across directories, scores become partly random —
   and random noise on a leaderboard looks like a small improvement.

Red-teaming evaluation means running these three against your own numbers before
someone else does.

## 3 · Demo — build a harness with zero capability

In [ ]:
import json
from dataclasses import dataclass

@dataclass
class Truth:
    qid: str; cwe: str; file: str

def make_corpus(n=40, skew=0.8):
    """A corpus where `skew` of the answers are one class — very common."""
    truths = {}
    n_major = int(n * skew)
    for i in range(1, n + 1):
        cwe = "CWE-89" if i <= n_major else ["CWE-78", "CWE-22", "CWE-798"][i % 3]
        truths[f"q{i}"] = Truth(f"q{i}", cwe, f"{cwe}/{i}.py")
    return truths

SKEWED = make_corpus(40, skew=0.8)
from collections import Counter
print("class balance:", Counter(t.cwe for t in SKEWED.values()))

class NullHarness:
    """No capability whatsoever. Emits perfect JSON and always guesses CWE-89."""
    def answer(self, qid, truth):
        return json.dumps({"qid": qid, "cwe": "CWE-89", "file": truth.file,
                           "line": 1, "rationale": "user input is concatenated"})

null = NullHarness()
ANSWERS = {q: null.answer(q, t) for q, t in SKEWED.items()}

In [ ]:
def path_key(p):
    parts = [x for x in p.replace("\\", "/").split("/") if x not in ("", ".")]
    return "/".join(parts[-2:]) if len(parts) > 1 else (parts[-1] if parts else "")
def basename(p): return p.replace("\\", "/").split("/")[-1]

def evaluate(answers, truths, matcher=path_key):
    conforming = expert = 0
    for qid, t in truths.items():
        try:
            d = json.loads(answers[qid])
        except json.JSONDecodeError:
            continue
        conforming += 1
        if matcher(d["file"]) != matcher(t.file):
            continue
        expert += 1.0 if d["cwe"].upper() == t.cwe else 0.5
    return {"conformance": conforming / len(truths),
            "expert_accuracy": expert / len(truths)}

r = evaluate(ANSWERS, SKEWED)
print("the null harness, scored on the skewed corpus:")
print(f"   conformance      {r['conformance']:.2f}   ← quotable as '100%'")
print(f"   expert accuracy  {r['expert_accuracy']:.2f}")
print("\nZero capability. Both numbers look like a working product.")

## 4 · Exploit 2 and 3 — balance the corpus, then break the matcher

In [ ]:
BALANCED = {}
for i in range(1, 41):
    cwe = ["CWE-89", "CWE-78", "CWE-22", "CWE-798"][i % 4]
    BALANCED[f"q{i}"] = Truth(f"q{i}", cwe, f"{cwe}/{i}.py")
ANS_B = {q: null.answer(q, t) for q, t in BALANCED.items()}

print("same null harness:")
for label, truths, ans in (("skewed corpus (80% CWE-89)", SKEWED, ANSWERS),
                           ("balanced corpus", BALANCED, ANS_B)):
    r = evaluate(ans, truths)
    print(f"   {label:28s} conformance {r['conformance']:.2f}  "
          f"expert accuracy {r['expert_accuracy']:.2f}")
print("\nBalancing the corpus removed most of the fake score. Nothing about the")
print("harness changed.")

In [ ]:
# exploit 3: a matcher that compares bare filenames.
# Build answers that point at the WRONG directory but the right filename.
WRONG_DIR = {}
for q, t in BALANCED.items():
    n = q[1:]
    WRONG_DIR[q] = json.dumps({"qid": q, "cwe": t.cwe,
                               "file": f"CWE-89/{n}.py",     # wrong dir, right basename
                               "line": 1, "rationale": "untrusted input"})

for matcher, name in ((path_key, "path_key (parent + filename)"),
                      (basename, "basename only (the bug)")):
    r = evaluate(WRONG_DIR, BALANCED, matcher)
    print(f"{name:34s} expert accuracy {r['expert_accuracy']:.2f}")
print("\nEvery answer names the wrong file. The basename matcher scores them")
print("as correct, because the corpus reuses numeric filenames across directories.")

## 5 · The control — a benchmark checklist you run on yourself

In [ ]:
def audit_benchmark(truths, answers, matcher):
    from collections import Counter
    counts = Counter(t.cwe for t in truths.values())
    majority_share = max(counts.values()) / len(truths)
    always_majority = {q: json.dumps({"qid": q, "cwe": counts.most_common(1)[0][0],
                                      "file": t.file, "line": 1, "rationale": "x"})
                       for q, t in truths.items()}
    floor = evaluate(always_majority, truths, matcher)["expert_accuracy"]
    real  = evaluate(answers, truths, matcher)
    collisions = len(truths) - len({matcher(t.file) for t in truths.values()})
    return {
      "majority_class_share": round(majority_share, 2),
      "score_of_always_guessing_majority": round(floor, 2),
      "reported_expert_accuracy": round(real["expert_accuracy"], 2),
      "lift_over_trivial_baseline": round(real["expert_accuracy"] - floor, 2),
      "matcher_collisions": collisions,
      "conformance_reported_as_quality": real["expert_accuracy"] < real["conformance"] - 0.2,
    }

print("audit of the skewed benchmark with a basename matcher:")
for k, v in audit_benchmark(SKEWED, ANSWERS, basename).items():
    print(f"   {k:38s} {v}")
print("\naudit of the balanced benchmark with path_key:")
for k, v in audit_benchmark(BALANCED, ANS_B, path_key).items():
    print(f"   {k:38s} {v}")

a = audit_benchmark(BALANCED, ANS_B, path_key)
assert a["lift_over_trivial_baseline"] <= 0.01
print("\nThe null harness has ~zero lift over the trivial baseline, which is the")
print("only honest way to describe it.")

## What you just proved

On the skewed corpus the zero-capability harness scores conformance 1.00 and expert accuracy around 0.85. Balancing the corpus drops expert accuracy to roughly 0.25. Answers naming the wrong directory score near zero under `path_key` and near 1.00 under a basename matcher. The audit reports the majority-class share, the trivial baseline, the matcher collisions, and near-zero lift.

## Your turn

Run the audit against a benchmark result your organisation relies on. Two questions decide it: what does always guessing the majority class score, and does the matcher collide? Most published numbers answer neither.

---

**Next → [C1.4 · Reporting agentic findings](https://spbreed.github.io/cyber-commons/lessons/C1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*